# Market Basket Analysis

Finds association rules between **product categories** purchased within the same order.

**Metrics reported**
- **Support** – fraction of orders that contain the itemset
- **Confidence** – P(consequent | antecedent)
- **Lift** – how much more likely the consequent is given the antecedent vs. by chance

Results are sorted by **confidence (descending)**.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from itertools import combinations

In [3]:
# ── Load data ──────────────────────────────────────────────────────────────────
orders      = pd.read_parquet("../EDA/outputs/orders.parquet")
lines       = pd.read_parquet("../EDA/outputs/lines.parquet")
cust        = pd.read_parquet("../EDA/outputs/customers.parquet")
rc_checkout = pd.read_parquet("../EDA/outputs/rc_checkout.parquet")
rc_recurring= pd.read_parquet("../EDA/outputs/rc_recurring.parquet")

orders["order_date"] = pd.to_datetime(orders["order_date"], utc=True)
lines["order_date"]  = pd.to_datetime(lines["order_date"],  utc=True)

print(f"Orders: {orders.shape[0]:,} rows")
print(f"Lines:  {lines.shape[0]:,} rows")

Orders: 27,350 rows
Lines:  50,963 rows


In [4]:
# ── Build basket: one row per order, one column per product category ────────────
# Each cell = 1 if that category appeared in the order, else 0

basket = (
    lines
    .groupby(["order_id", "product_category"])["Line: Quantity"]
    .sum()
    .unstack(fill_value=0)
    .reset_index(drop=True)
)

# Binarise: any quantity > 0 → True
basket_bool = basket.map(lambda x: True if x > 0 else False)

print(f"Basket shape: {basket_bool.shape}")
print(f"Product categories ({basket_bool.shape[1]}):")
print(list(basket_bool.columns))

Basket shape: (27901, 7)
Product categories (7):
['Accessories', 'Clear Protein', 'Collagen Glow', 'Lean Protein', 'Other', 'Soy Protein', 'Unknown']


In [5]:
# ── Apriori — frequent itemsets ────────────────────────────────────────────────
# min_support = 0.01 (1% of orders).  Lower if you have few categories.

MIN_SUPPORT = 0.01

frequent_itemsets = apriori(
    basket_bool,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=None,          # no cap on itemset size
)

frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found: {len(frequent_itemsets):,}")
frequent_itemsets.sort_values("support", ascending=False).head(20)

Frequent itemsets found: 17


,support,itemsets,length
6,0.512957,(Unknown),1
4,0.266800,(Other),1
3,0.134762,(Lean Protein),1
1,0.133615,(Clear Protein),1
0,0.081538,(Accessories),1
2,0.068743,(Collagen Glow),1
5,0.034587,(Soy Protein),1
16,0.028888,"(Unknown, Other)",2
9,0.027813,"(Other, Accessories)",2
8,0.027741,"(Lean Protein, Accessories)",2


In [6]:
# ── Association rules — ALL pairs, sorted by confidence desc ──────────────────

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0,     # keep every rule; filter below if needed
    num_itemsets=len(frequent_itemsets),
)

# Pretty-print antecedent / consequent as strings
rules["antecedents"] = rules["antecedents"].apply(lambda x: " + ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: " + ".join(sorted(x)))

# Keep only the metrics we care about and sort by confidence
rules_out = (
    rules[[
        "antecedents", "consequents",
        "support", "confidence", "lift",
        "leverage", "conviction", "zhangs_metric",
    ]]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

print(f"Total rules generated: {len(rules_out):,}")
rules_out.head(20)

Total rules generated: 20


,antecedents,consequents,support,confidence,lift,leverage,conviction,zhangs_metric
0,Collagen Glow,Other,0.023870,0.347237,1.301485,0.005529,1.123224,0.248746
1,Accessories,Other,0.027813,0.341099,1.278479,0.006058,1.112761,0.237158
2,Accessories,Lean Protein,0.027741,0.340220,2.524594,0.016753,1.311403,0.657509
3,Accessories,Clear Protein,0.026307,0.322637,2.414674,0.015413,1.279056,0.637877
4,Lean Protein,Accessories,0.027741,0.205851,2.524594,0.016753,1.156536,0.697955
5,Clear Protein,Lean Protein,0.027454,0.205472,1.524701,0.009448,1.088996,0.397207
6,Lean Protein,Clear Protein,0.027454,0.203723,1.524701,0.009448,1.088045,0.397733
7,Clear Protein,Accessories,0.026307,0.196888,2.414674,0.015413,1.143629,0.676219
8,Lean Protein,Other,0.024766,0.183777,0.688817,-0.011188,0.898283,-0.343025
9,Clear Protein,Other,0.023332,0.174624,0.654513,-0.012316,0.888322,-0.378596


In [7]:
# ── Filter: PAIR rules only (1 antecedent → 1 consequent) ─────────────────────

pair_rules = rules_out[
    rules_out["antecedents"].str.count(r" \+ ").eq(0) &   # no " + " = single item
    rules_out["consequents"].str.count(r" \+ ").eq(0)
].copy()

print(f"Pair rules (1→1): {len(pair_rules):,}")
pair_rules

Pair rules (1→1): 20


,antecedents,consequents,support,confidence,lift,leverage,conviction,zhangs_metric
0,Collagen Glow,Other,0.023870,0.347237,1.301485,0.005529,1.123224,0.248746
1,Accessories,Other,0.027813,0.341099,1.278479,0.006058,1.112761,0.237158
2,Accessories,Lean Protein,0.027741,0.340220,2.524594,0.016753,1.311403,0.657509
3,Accessories,Clear Protein,0.026307,0.322637,2.414674,0.015413,1.279056,0.637877
4,Lean Protein,Accessories,0.027741,0.205851,2.524594,0.016753,1.156536,0.697955
5,Clear Protein,Lean Protein,0.027454,0.205472,1.524701,0.009448,1.088996,0.397207
6,Lean Protein,Clear Protein,0.027454,0.203723,1.524701,0.009448,1.088045,0.397733
7,Clear Protein,Accessories,0.026307,0.196888,2.414674,0.015413,1.143629,0.676219
8,Lean Protein,Other,0.024766,0.183777,0.688817,-0.011188,0.898283,-0.343025
9,Clear Protein,Other,0.023332,0.174624,0.654513,-0.012316,0.888322,-0.378596


In [8]:
# ── Manual pair computation (exhaustive, no support floor) ────────────────────
# This section computes ALL category pairs directly from the basket,
# bypassing any min_support filter so you never miss a low-frequency pair.

n_orders = len(basket_bool)
cats     = list(basket_bool.columns)

rows = []
for a, b in combinations(cats, 2):
    sup_a  = basket_bool[a].sum() / n_orders
    sup_b  = basket_bool[b].sum() / n_orders
    sup_ab = (basket_bool[a] & basket_bool[b]).sum() / n_orders

    if sup_a == 0 or sup_b == 0 or sup_ab == 0:
        continue

    conf_ab = sup_ab / sup_a          # P(B|A)
    conf_ba = sup_ab / sup_b          # P(A|B)
    lift    = sup_ab / (sup_a * sup_b)

    rows.append({
        "antecedent":  a, "consequent":  b,
        "support":     round(sup_ab, 6),
        "confidence":  round(conf_ab, 6),
        "lift":        round(lift, 4),
    })
    rows.append({
        "antecedent":  b, "consequent":  a,
        "support":     round(sup_ab, 6),
        "confidence":  round(conf_ba, 6),
        "lift":        round(lift, 4),
    })

all_pairs = (
    pd.DataFrame(rows)
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

print(f"All directional pairs: {len(all_pairs):,}")
all_pairs

All directional pairs: 42


,antecedent,consequent,support,confidence,lift
0,Collagen Glow,Other,0.023870,0.347237,1.3015
1,Accessories,Other,0.027813,0.341099,1.2785
2,Accessories,Lean Protein,0.027741,0.340220,2.5246
3,Accessories,Clear Protein,0.026307,0.322637,2.4147
4,Soy Protein,Other,0.008817,0.254922,0.9555
5,Lean Protein,Accessories,0.027741,0.205851,2.5246
6,Clear Protein,Lean Protein,0.027454,0.205472,1.5247
7,Lean Protein,Clear Protein,0.027454,0.203723,1.5247
8,Clear Protein,Accessories,0.026307,0.196888,2.4147
9,Lean Protein,Other,0.024766,0.183777,0.6888


In [9]:
# ── Quick summary: top rules by lift (positive associations) ──────────────────

print("=== Top 10 pairs by LIFT (strongest positive associations) ===")
display(
    all_pairs.drop_duplicates(subset=["support"])  # unique pairs (undirected)
    .sort_values("lift", ascending=False)
    .head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
    .reset_index(drop=True)
)

print("\n=== Top 10 pairs by CONFIDENCE (most predictable) ===")
display(
    all_pairs.head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
)

print("\n=== Top 10 pairs by SUPPORT (most common co-purchases) ===")
display(
    all_pairs.drop_duplicates(subset=["support"])
    .sort_values("support", ascending=False)
    .head(10)
    [["antecedent", "consequent", "support", "confidence", "lift"]]
    .reset_index(drop=True)
)

=== Top 10 pairs by LIFT (strongest positive associations) ===


,antecedent,consequent,support,confidence,lift
0,Accessories,Lean Protein,0.027741,0.340220,2.5246
1,Accessories,Clear Protein,0.026307,0.322637,2.4147
2,Clear Protein,Lean Protein,0.027454,0.205472,1.5247
3,Soy Protein,Collagen Glow,0.003154,0.091192,1.3266
4,Collagen Glow,Other,0.023870,0.347237,1.3015
5,Accessories,Other,0.027813,0.341099,1.2785
6,Collagen Glow,Accessories,0.006631,0.096455,1.1829
7,Collagen Glow,Clear Protein,0.009820,0.142857,1.0692
8,Collagen Glow,Lean Protein,0.009641,0.140250,1.0407
9,Soy Protein,Accessories,0.002724,0.078756,0.9659



=== Top 10 pairs by CONFIDENCE (most predictable) ===


,antecedent,consequent,support,confidence,lift
0,Collagen Glow,Other,0.023870,0.347237,1.3015
1,Accessories,Other,0.027813,0.341099,1.2785
2,Accessories,Lean Protein,0.027741,0.340220,2.5246
3,Accessories,Clear Protein,0.026307,0.322637,2.4147
4,Soy Protein,Other,0.008817,0.254922,0.9555
5,Lean Protein,Accessories,0.027741,0.205851,2.5246
6,Clear Protein,Lean Protein,0.027454,0.205472,1.5247
7,Lean Protein,Clear Protein,0.027454,0.203723,1.5247
8,Clear Protein,Accessories,0.026307,0.196888,2.4147
9,Lean Protein,Other,0.024766,0.183777,0.6888



=== Top 10 pairs by SUPPORT (most common co-purchases) ===


,antecedent,consequent,support,confidence,lift
0,Other,Unknown,0.028888,0.108275,0.2111
1,Accessories,Other,0.027813,0.341099,1.2785
2,Accessories,Lean Protein,0.027741,0.340220,2.5246
3,Clear Protein,Lean Protein,0.027454,0.205472,1.5247
4,Accessories,Clear Protein,0.026307,0.322637,2.4147
5,Lean Protein,Other,0.024766,0.183777,0.6888
6,Collagen Glow,Other,0.023870,0.347237,1.3015
7,Clear Protein,Other,0.023332,0.174624,0.6545
8,Accessories,Unknown,0.012795,0.156923,0.3059
9,Clear Protein,Unknown,0.010107,0.075644,0.1475


In [10]:
# ── Export ─────────────────────────────────────────────────────────────────────
# OUTPUT_DIR = "../EDA/outputs"   # adjust to your OUTPUT_DIR if needed

# rules_out.to_csv(f"{OUTPUT_DIR}/05_mba_all_rules.csv",  index=False)
# all_pairs.to_csv(f"{OUTPUT_DIR}/05_mba_all_pairs.csv",  index=False)
# pair_rules.to_csv(f"{OUTPUT_DIR}/05_mba_pair_rules.csv", index=False)

# print("Saved:")
# print(f"  05_mba_all_rules.csv  — {len(rules_out):,} rules (all lengths, apriori)")
# print(f"  05_mba_all_pairs.csv  — {len(all_pairs):,} directional pairs (manual, no support floor)")
# print(f"  05_mba_pair_rules.csv — {len(pair_rules):,} 1→1 pair rules (apriori)")
# print("\n[05] Market Basket Analysis done.")